# Quantum vs Classical GNN - OPTIMIZED FOR OVERNIGHT

**Optimizations:**
- 2 quantum layers (instead of 3) → 30% faster
- 1000 drugs (instead of 2000) → 50% faster
- Batch size 128 → Faster feedback

**Expected time: 8-10 hours** ✅

In [1]:
# OPTIMIZED Configuration for Overnight Run
import os
os.environ['OMP_NUM_THREADS'] = '16'
os.environ['MKL_NUM_THREADS'] = '16'

DATA_DIR = "/media/priyanshu/SD/othercode/data"
SAVE_DIR = "./optimized_comparison_results"

# Data parameters - REDUCED FOR SPEED
MAX_DRUGS = 1000              # ⚡ Half the data = 50% faster
SEED = 42

# Model parameters - OPTIMIZED
NUM_QUBITS = 6                # Keep at 6 for expressivity
NUM_QLAYERS = 2               # ⚡ Reduced from 3 = 30% faster
HIDDEN_DIM = 128              # Keep at 128

# Training parameters
EPOCHS = 100
BATCH_SIZE = 128              # ⚡ Smaller = faster iterations
LEARNING_RATE_QUANTUM = 0.005
LEARNING_RATE_CLASSICAL = 0.0005
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 15

# Hardware - FIXED FOR GPU
DEVICE = 'cuda'
QUANTUM_DEVICE = 'lightning.gpu'  # GPU quantum device
NUM_WORKERS = 0  # IMPORTANT: Set to 0 to avoid multiprocessing issues with quantum circuits
PIN_MEMORY = True
VERBOSE = 1

print(f"⚡ OPTIMIZED Configuration:")
print(f"  Dataset: {MAX_DRUGS} drugs (reduced for speed)")
print(f"  Quantum layers: {NUM_QLAYERS} (reduced from 3)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Device: {DEVICE}")
print(f"  Quantum Device: {QUANTUM_DEVICE}")
print(f"  Workers: {NUM_WORKERS} (avoiding multiprocessing issues)")
print(f"  Expected time: 8-10 hours")

⚡ OPTIMIZED Configuration:
  Dataset: 1000 drugs (reduced for speed)
  Quantum layers: 2 (reduced from 3)
  Batch size: 128
  Device: cuda
  Quantum Device: lightning.gpu
  Workers: 0 (avoiding multiprocessing issues)
  Expected time: 8-10 hours


In [2]:
import importlib
import drug_patient_qgnn.data_processing
import drug_patient_qgnn

importlib.reload(drug_patient_qgnn.data_processing)
importlib.reload(drug_patient_qgnn)

%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm  # Progress bars!

from drug_patient_qgnn import (
    DrugPatientDataProcessor,
    QuantumDrugPatientGNN,
    set_seed,
    print_model_summary,
    print_device_info
)

set_seed(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)

print_device_info()
print(f"\nStarting optimized run at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Device Information
CUDA Available      : True
CUDA Devices        : 1
CUDA Device Name    : NVIDIA GeForce RTX 3080
MPS Available       : False


Starting optimized run at: 2025-12-01 09:21:26


In [3]:
print(f"Loading REDUCED dataset ({MAX_DRUGS} drugs)...\n")
start_time = datetime.now()

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)

stats = processor.get_statistics()
print(f"\nData loaded in {(datetime.now() - start_time).total_seconds():.1f}s")
print("\nDataset Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key:25s}: {value:.4f}")
    else:
        print(f"  {key:25s}: {value}")

Loading REDUCED dataset (1000 drugs)...

Searching for data in: /media/priyanshu/SD/othercode/data
Found 1000 protein descriptor files


Loading PDB Data: 100%|██████████| 1000/1000 [00:04<00:00, 236.80it/s]


Generating negative samples (target: 7264)...


Generating Negatives: 100%|██████████| 7264/7264 [00:00<00:00, 307162.26it/s]

Loaded 855 proteins, 7264 drugs
Interactions: 7264 positive, 7264 negative (Total: 14528)

Data loaded in 102.0s

Dataset Statistics:
  num_ligands              : 3662
  num_pockets              : 822
  num_interactions         : 14528
  num_drugs                : 3662
  num_patients             : 822
  positive_rate            : 0.5000
  negative_rate            : 0.5000
  ligand_feature_dim       : 11
  drug_feature_dim         : 11
  pocket_feature_dim       : 19
  patient_feature_dim      : 19


In [4]:
# Get graph data
graph = processor.graph
drug_features = graph.get_drug_features_matrix()
patient_features = graph.get_patient_features_matrix()
edge_index, edge_features = graph.get_edge_index()
labels = graph.get_edge_labels()

# Create interaction dataset
interaction_data = []
for idx in range(edge_index.shape[1]):
    drug_idx = int(edge_index[0, idx])
    patient_idx = int(edge_index[1, idx])
    
    interaction_data.append({
        'drug_features': drug_features[drug_idx].tolist(),
        'patient_features': patient_features[patient_idx].tolist(),
        'label': float(labels[idx])
    })

df_pandas = pd.DataFrame(interaction_data)

# Stratified split
train_pd, val_pd = train_test_split(
    df_pandas,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=df_pandas['label']
)

batches_per_epoch = len(train_pd) // BATCH_SIZE
estimated_time_per_epoch = batches_per_epoch * 12 / 60  # 12s per batch estimate

print(f"\nTraining samples:   {len(train_pd):,}")
print(f"Validation samples: {len(val_pd):,}")
print(f"Batches per epoch:  {batches_per_epoch}")
print(f"Estimated time/epoch: {estimated_time_per_epoch:.1f} minutes")
print(f"Estimated total (50 epochs): {estimated_time_per_epoch * 50 / 60:.1f} hours")

# PyTorch Dataset
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.drug_features = np.stack(df['drug_features'].values)
        self.patient_features = np.stack(df['patient_features'].values)
        self.labels = df['label'].values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.drug_features[idx], dtype=torch.float32),
            torch.tensor(self.patient_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

train_dataset = InteractionDataset(train_pd)
val_dataset = InteractionDataset(val_pd)

# Create DataLoaders with no multiprocessing (NUM_WORKERS=0) to avoid quantum circuit pickling issues
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"\n✓ DataLoaders ready (workers={NUM_WORKERS}, pin_memory={PIN_MEMORY})")


Training samples:   11,622
Validation samples: 2,906
Batches per epoch:  90
Estimated time/epoch: 18.0 minutes
Estimated total (50 epochs): 15.0 hours

✓ DataLoaders ready (workers=0, pin_memory=True)


In [5]:
def train_epoch(model, optimizer, criterion, loader, device, show_progress=True):
    """Train one epoch WITH PROGRESS BAR."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    # PROGRESS BAR FOR DEBUGGING
    pbar = tqdm(loader, desc='Training', disable=not show_progress)
    
    for batch_idx, (drug_features, patient_features, labels) in enumerate(pbar):
        batch_start = datetime.now()
        
        drug_features = drug_features.to(device, non_blocking=True)
        patient_features = patient_features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        outputs = model(drug_features, patient_features).squeeze(-1)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar with batch time
        batch_time = (datetime.now() - batch_start).total_seconds()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'batch_time': f'{batch_time:.1f}s'})
        
        # Optional verbose logging for deeper debugging
        if VERBOSE >= 2 and (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)}: loss={loss.item():.4f}, time={batch_time:.1f}s")
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}


def evaluate(model, criterion, loader, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for drug_features, patient_features, labels in tqdm(loader, desc='Validation', leave=False):
            drug_features = drug_features.to(device, non_blocking=True)
            patient_features = patient_features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(drug_features, patient_features).squeeze(-1)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }


def train_model(model, train_loader, val_loader, learning_rate, model_name, device):
    """Train with detailed progress tracking."""
    print(f"\n{'='*70}")
    print(f"Training {model_name.upper()} Model")
    print(f"{'='*70}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Batches per epoch: {len(train_loader)}")
    print(f"Estimated time/epoch: ~{len(train_loader) * 2 / 60:.1f} min\n")
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    criterion = torch.nn.BCEWithLogitsLoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': [],
        'learning_rates': []
    }
    
    best_val_auc = 0.0
    patience_counter = 0
    start_time = datetime.now()
    
    for epoch in range(EPOCHS):
        epoch_start = datetime.now()
        print(f"\n{'='*70}")
        print(f"EPOCH {epoch+1}/{EPOCHS} - Started at {epoch_start.strftime('%H:%M:%S')}")
        print(f"{'='*70}")
        
        train_metrics = train_epoch(model, optimizer, criterion, train_loader, device)
        val_metrics = evaluate(model, criterion, val_loader, device)
        
        # Update scheduler
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_metrics['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        
        if current_lr != old_lr:
            print(f"  Learning rate reduced: {old_lr:.6f} → {current_lr:.6f}")
        
        # Save metrics
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        history['learning_rates'].append(current_lr)
        
        epoch_time = (datetime.now() - epoch_start).total_seconds()
        
        # Print results
        print(f"\n{'-'*70}")
        print(f"Epoch {epoch+1} Results ({epoch_time:.1f}s):")
        print(f"  Train: loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}")
        print(f"  Val:   loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
              f"auc={val_metrics['auc']:.4f}, f1={val_metrics['f1']:.4f}")
        print(f"  Best AUC so far: {best_val_auc:.4f}")
        print(f"{'-'*70}")
        
        # Save best model
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_val_auc,
                'history': history
            }, os.path.join(SAVE_DIR, f"{model_name}_best.pt"))
            
            print(f"✓ New best AUC: {best_val_auc:.4f} (saved)")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
        
        # Auto-save every epoch for debugging
        with open(os.path.join(SAVE_DIR, f"{model_name}_history.json"), 'w') as f:
            json.dump(history, f, indent=2)
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    total_time = (datetime.now() - start_time).total_seconds()
    print(f"\n{model_name.upper()} Training Complete!")
    print(f"  Total time: {total_time/3600:.2f} hours")
    print(f"  Best AUC: {best_val_auc:.4f}")
    
    return history, best_val_auc

## Train Quantum Model

In [6]:
drug_dim = len(drug_features[0])
patient_dim = len(patient_features[0])

print("Creating Quantum Model (OPTIMIZED: 2 layers)...")
quantum_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,
    hidden_dim=HIDDEN_DIM,
    use_quantum=True,
    device_name=QUANTUM_DEVICE
)

print_model_summary(quantum_model, drug_dim, patient_dim)

print(f"\n🚀 Quantum training started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Expected completion: ~{datetime.now().hour + 8}:00 ({8} hours from now)\n")

quantum_history, quantum_best_auc = train_model(
    quantum_model,
    train_loader,
    val_loader,
    LEARNING_RATE_QUANTUM,
    "quantum",
    DEVICE
)

print(f"\n✅ Quantum training finished at: {datetime.now().strftime('%H:%M:%S')}")

Creating Quantum Model (OPTIMIZED: 2 layers)...

Model Summary
ligand_dim          : 11
pocket_dim          : 19
num_qubits          : 6
num_qlayers         : 2
use_quantum         : True
num_parameters      : 6201
drug_dim            : 11
patient_dim         : 19
Total parameters    : 6,201
Trainable params    : 6,201
Input (drug)        : (11,)
Input (patient)     : (19,)
Output              : (1,) [probability]


🚀 Quantum training started at: 2025-12-01 09:23:08
Expected completion: ~17:00 (8 hours from now)


Training QUANTUM Model
Learning Rate: 0.005
Device: cuda
Parameters: 6,201
Batches per epoch: 91
Estimated time/epoch: ~3.0 min


EPOCH 1/100 - Started at 09:23:09


Training: 100%|██████████| 91/91 [27:34<00:00, 18.18s/it, loss=0.6926, batch_time=14.7s]



----------------------------------------------------------------------
Epoch 1 Results (1752.6s):
  Train: loss=0.6780, acc=0.5661
  Val:   loss=0.6674, acc=0.5843, auc=0.6046, f1=0.6317
  Best AUC so far: 0.0000
----------------------------------------------------------------------
✓ New best AUC: 0.6046 (saved)

EPOCH 2/100 - Started at 09:52:22


Training:  27%|██▋       | 25/91 [07:49<20:38, 18.77s/it, loss=0.7096, batch_time=17.8s]


KeyboardInterrupt: 

## Train Classical Model

In [ ]:
print("Creating Classical Model...")
classical_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,
    hidden_dim=HIDDEN_DIM,
    use_quantum=False
)

print_model_summary(classical_model, drug_dim, patient_dim)

print(f"\n🚀 Classical training started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

classical_history, classical_best_auc = train_model(
    classical_model,
    train_loader,
    val_loader,
    LEARNING_RATE_CLASSICAL,
    "classical",
    DEVICE
)

print(f"\n✅ Classical training finished at: {datetime.now().strftime('%H:%M:%S')}")

## Results

In [ ]:
# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(quantum_history['val_loss'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[0, 0].plot(classical_history['val_loss'], label='Classical', linewidth=2.5, color='#A23B72')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Validation Loss')
axes[0, 0].set_title('Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(quantum_history['val_auc'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[0, 1].plot(classical_history['val_auc'], label='Classical', linewidth=2.5, color='#A23B72')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Validation AUC-ROC')
axes[0, 1].set_title('⭐ Validation AUC-ROC (Main Metric)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(quantum_history['val_acc'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[1, 0].plot(classical_history['val_acc'], label='Classical', linewidth=2.5, color='#A23B72')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Validation Accuracy')
axes[1, 0].set_title('Validation Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(quantum_history['val_f1'], label='Quantum', linewidth=2.5, color='#2E86AB')
axes[1, 1].plot(classical_history['val_f1'], label='Classical', linewidth=2.5, color='#A23B72')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Validation F1 Score')
axes[1, 1].set_title('Validation F1 Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quantum_vs_classical_comparison.png'), dpi=300)
plt.show()

print(f"\n✓ Plot saved to: {SAVE_DIR}/quantum_vs_classical_comparison.png")

In [ ]:
print("\n" + "="*80)
print("🏆 FINAL RESULTS: QUANTUM vs CLASSICAL GNN (OPTIMIZED)")
print("="*80)

print(f"\n{'Metric':<25} {'Quantum':<15} {'Classical':<15} {'Difference':<15} {'Winner'}")
print("-"*80)

metrics = [
    ('Best Validation AUC', quantum_best_auc, classical_best_auc),
    ('Final Val Accuracy', quantum_history['val_acc'][-1], classical_history['val_acc'][-1]),
    ('Final Val F1 Score', quantum_history['val_f1'][-1], classical_history['val_f1'][-1]),
]

quantum_wins = 0
for metric_name, quantum_val, classical_val in metrics:
    diff = quantum_val - classical_val
    diff_pct = (diff / classical_val) * 100
    winner = '🏆 QUANTUM' if quantum_val > classical_val else '🏆 Classical'
    if quantum_val > classical_val:
        quantum_wins += 1
    print(f"{metric_name:<25} {quantum_val:<15.4f} {classical_val:<15.4f} {diff:+.4f} ({diff_pct:+.1f}%)  {winner}")

print("\n" + "="*80)
print(f"Quantum wins: {quantum_wins}/3 metrics")
print("="*80)

# Save results
results = {
    'quantum': {'best_auc': quantum_best_auc, 'history': quantum_history},
    'classical': {'best_auc': classical_best_auc, 'history': classical_history},
    'config': {
        'max_drugs': MAX_DRUGS,
        'num_qubits': NUM_QUBITS,
        'num_qlayers': NUM_QLAYERS,
        'batch_size': BATCH_SIZE
    }
}

with open(os.path.join(SAVE_DIR, 'final_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to: {SAVE_DIR}/final_results.json")
print(f"\n🎉 EXPERIMENT COMPLETE at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")